In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from sentence_transformers import SentenceTransformer

d:\projects\GENAi\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
loader = PyPDFLoader("../Data/mayank.pdf")
docs = loader.load()

In [5]:
len(docs)

43

In [7]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1500,chunk_overlap=350)
splitted_data = splitter.split_documents(docs)
len(splitted_data)

55

In [14]:
embeddings = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3623.88it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

C:\Users\mkdog\AppData\Local\Temp\ipykernel_7076\494109701.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8637.50it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [21]:
vector_store = Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings

)

In [26]:
query = """The  key  challenge  in  building  an  efficient  fraud  detection  model  lies  in  handling  the 
imbalanced nature of the data"""
data = vector_store.similarity_search(query=query)
len(data)

4

In [27]:
data[0]

Document(id='e1545e12-6695-4641-b716-9f4e4be0196f', metadata={'moddate': '2025-11-18T20:26:39+05:30', 'author': 'Mayank Dogra', 'page_label': '9', 'creationdate': '2025-11-18T20:26:39+05:30', 'total_pages': 43, 'creator': 'Microsoft® Word 2021', 'source': '../Data/mayank.pdf', 'page': 8, 'producer': 'Microsoft® Word 2021'}, page_content='reliable and robust, especially for classification problems like fraud detection \nTo evaluate the effectiveness of the model, performance metrics such as confusion matrix, \nprecision, recall, and F1-score are used. These metrics help in understanding how well the \nmodel performs, particularly in identifying true fraudulent transactions (recall) without \nmisclassifying too many legitimate transactions as fraud (precision). The confusion matrix \nprovides a clear view of the number of true positives, false positives, true negatives, and false \nnegatives, which is crucial in fraud detection scenarios. \nThe key challenge in building an efficient frau

In [28]:
context = ""
for doc in data:
    context += doc.page_content +"\n"
print(context)

reliable and robust, especially for classification problems like fraud detection 
To evaluate the effectiveness of the model, performance metrics such as confusion matrix, 
precision, recall, and F1-score are used. These metrics help in understanding how well the 
model performs, particularly in identifying true fraudulent transactions (recall) without 
misclassifying too many legitimate transactions as fraud (precision). The confusion matrix 
provides a clear view of the number of true positives, false positives, true negatives, and false 
negatives, which is crucial in fraud detection scenarios. 
The key challenge in building an efficient fraud detection model lies in handling the 
imbalanced nature of the data. In this project, SMOTE plays a crucial role in mitigating this 
challenge by generating synthetic examples of the minority class, thereby helping the model 
learn patterns in fraudulent behavior more effectively. The combination of SMOTE with 
Random Forest results in a balan

In [29]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

In [30]:
res = llm.invoke(f"""
    can u provide me the answer based on context :{context} from question:{query}
""")

In [31]:
print(res.content)

The key challenge in building an efficient fraud detection model lies in **handling the imbalanced nature of the data**.


In [37]:
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in data:
        context += doc.page_content +"\n"

        return{
            "context" :context,
            "question":query
        }
    

In [38]:
from langchain_core.prompts import PromptTemplate

In [39]:
prompt = PromptTemplate.from_template("""
    You are a helpful assistant and provide answer based on the context for user question.
    and if you dont know the answer you simply say" I dont know about this"
    context:{context}
    Question:{question}
""")

In [40]:
rag_chain = get_context | prompt |llm 

In [55]:
res_new = rag_chain.invoke( """I hereby declare that the project work entitled “FraudShield – Smart Fraud Detection” 
submitted in """)

In [56]:
print(res_new.content)

I hereby declare that the project work entitled “FraudShield – Smart Fraud Detection” submitted in partial fulfillment of the requirements for the award of the degree of Bachelor of Technology in Computer Science Engineering at Aarni University, Indora, is my original work and has not been submitted earlier to any other University or Institution for the award of any degree or diploma.
